# Examen 2
> Moctezuma Ramirez Diego Rafael

In [1]:
# Data Wrangling
import numpy as np
import pandas as pd

# Data viz
import plotly.express as px
from sklearn import set_config

# Data preprocessing
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Model performance
from sklearn.metrics import roc_auc_score

# Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

# Enviroment setup
set_config(display="diagram")
pd.set_option("display.max_columns", 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

## Funciones Auxiliares

In [2]:
def frecuencias(df, caracteristicas):
    for caracteristica in caracteristicas:
        print(f"Caracteristica: {caracteristica}")
        abs_ = df[caracteristica].value_counts(dropna=False).to_frame().rename(columns={"count": "Frecuencia absoluta"})
        rel_ = df[caracteristica].value_counts(dropna=False, normalize= True).to_frame().rename(columns={"proportion": "Frecuencia relativa"})
        freq = abs_.join(rel_)
        freq["Frecuencia Acumulada"] = freq["Frecuencia absoluta"].cumsum()
        freq["Acumulada %"] = freq["Frecuencia relativa"].cumsum()
        freq["Frecuencia absoluta"] = freq["Frecuencia absoluta"].map(lambda x: f"{x:,.0f}")
        freq["Frecuencia relativa"] = freq["Frecuencia relativa"].map(lambda x: f"{x:,.2%}")
        freq["Frecuencia Acumulada"] = freq["Frecuencia Acumulada"].map(lambda x: f"{x:,.0f}")
        freq["Acumulada %"] = freq["Acumulada %"].map(lambda x: f"{x:,.2%}")
        display(freq)

In [3]:
def IV(df, var, tgt):
    aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])
    aux["evento"] = aux[tgt, "sum"]
    aux["no_evento"] = aux[tgt, "count"] - aux[tgt, "sum"]
    aux["%evento"] = aux["evento"] / aux["evento"].sum()
    aux["%no_evento"] = aux["no_evento"] / aux["no_evento"].sum()
    aux["WOE"] = np.log(aux["%no_evento"] / aux["%evento"])
    aux["IV"] = (aux["%no_evento"] - aux["%evento"])*aux["WOE"]
    return aux["IV"].sum()

In [4]:
def WOE(df, var, tgt):
    aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])
    
    aux["evento"] = aux[tgt, "sum"]
    aux["no_evento"] = aux[tgt, "count"] - aux[tgt, "sum"]
    aux["%evento"] = aux["evento"] / aux["evento"].sum()
    aux["%no_evento"] = aux["no_evento"] / aux["no_evento"].sum()
    
    aux["WOE"] = np.log(aux["%no_evento"] / aux["%evento"])
    
    dic_woe = aux["WOE"].to_dict()
    display(aux)
    
    aux.columns = aux.columns.droplevel(1)
    aux = aux[["WOE"]].reset_index().rename(columns={"WOE": f"W_{var}"})
    
    df = df.merge(aux, on = var, how = "left")
    return df, dic_woe

## Carga de datos

In [5]:
df = pd.read_csv("Clasificacion/train_default.csv", sep="|")

In [6]:
df.sample(5)

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
3221,24834,200000.0000,2,2,1,30,1,-2,-1,0,0,0,26223.0000,-137.0000,73284.0000,65428.0000,65820.0000,2859.0000,0.0000,73421.0000,1309.0000,392.0000,2859.0000,1250.0000,0
5436,5964,370000.0000,2,2,1,47,-2,-2,-2,-2,-2,-2,1289.0000,2408.0000,395.0000,12074.0000,744.0000,4040.0000,2432.0000,397.0000,12133.0000,747.0000,4060.0000,3059.0000,0
5206,15168,50000.0000,1,2,2,28,0,0,0,0,0,0,17868.0000,18028.0000,38126.0000,18690.0000,19082.0000,19505.0000,1711.0000,20809.0000,668.0000,692.0000,740.0000,632.0000,0
5067,11210,240000.0000,2,2,2,51,-2,-2,-2,-2,-2,-2,489.0000,4006.0000,696.0000,696.0000,696.0000,696.0000,4006.0000,696.0000,696.0000,696.0000,696.0000,1542.0000,0
1805,8618,130000.0000,2,1,2,30,-1,-1,-1,-1,-1,-1,5002.0000,2437.0000,2448.0000,5457.0000,3855.0000,4684.0000,2437.0000,2774.0000,5457.0000,3855.0000,4684.0000,843.0000,0


In [7]:
df.dtypes

CUSTOMER_ID                     int64
LIMIT_BAL                     float64
SEX                             int64
EDUCATION                       int64
MARRIAGE                        int64
AGE                             int64
PAY_0                           int64
PAY_2                           int64
PAY_3                           int64
PAY_4                           int64
PAY_5                           int64
PAY_6                           int64
BILL_AMT1                     float64
BILL_AMT2                     float64
BILL_AMT3                     float64
BILL_AMT4                     float64
BILL_AMT5                     float64
BILL_AMT6                     float64
PAY_AMT1                      float64
PAY_AMT2                      float64
PAY_AMT3                      float64
PAY_AMT4                      float64
PAY_AMT5                      float64
PAY_AMT6                      float64
default.payment.next.month      int64
dtype: object

In [8]:
df.isna().sum()

CUSTOMER_ID                   0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64

## Filtrado de variables

In [9]:
ls_discretas = ["SEX","EDUCATION","MARRIAGE","PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
ls_continuas = ["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6","PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6","LIMIT_BAL","AGE"]
ls_drop = ["CUSTOMER_ID"]

target = "default.payment.next.month"

In [10]:
df.drop(columns=ls_drop, inplace=True)

## Credit Scoring

In [11]:
df.sample(5)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
1842,180000.0000,2,1,2,31,4,4,4,3,2,2,83020.0000,84654.0000,82852.0000,81162.0000,79958.0000,178148.0000,3500.0000,0.0000,0.0000,0.0000,100000.0000,0.0000,1
3866,10000.0000,1,2,1,38,0,0,2,0,0,2,6798.0000,8055.0000,7803.0000,8623.0000,9527.0000,9780.0000,1500.0000,0.0000,1100.0000,1000.0000,504.0000,0.0000,1
1959,350000.0000,2,2,1,35,-1,-1,2,-1,-1,-1,439.0000,884.0000,421.0000,267.0000,575.0000,932.0000,884.0000,0.0000,267.0000,575.0000,932.0000,0.0000,1
1040,50000.0000,2,1,0,45,-1,-1,-2,-2,-1,2,4736.0000,0.0000,0.0000,0.0000,4881.0000,4634.0000,0.0000,0.0000,0.0000,4881.0000,0.0000,0.0000,0
336,50000.0000,1,3,1,32,0,0,0,0,0,0,40992.0000,35215.0000,29156.0000,18891.0000,19443.0000,19478.0000,2026.0000,2021.0000,1000.0000,1001.0000,1000.0000,707.0000,0


### Discretización

In [12]:
for n_bins in range(2, 10):
    for caracteristica in ls_continuas:
        df[f"C_{caracteristica}"] = pd.qcut(df[caracteristica], q=n_bins, duplicates="drop").cat.add_categories(["Missing"]).fillna("Missing").astype(str)
    ls_discretas_C = [x for x in df.columns if x.startswith("C_")]
    df_iv = pd.DataFrame(columns=["iv"])
    for caracteristica in ls_discretas_C:
        df_iv.loc[caracteristica, "iv"] = IV(df = df, var = caracteristica, tgt = target)
    print(f"{n_bins} bins")
    display(df_iv)

2 bins


,iv
C_BILL_AMT1,0.0117
C_BILL_AMT2,0.0030
C_BILL_AMT3,0.0000
C_BILL_AMT4,0.0003
C_BILL_AMT5,0.0002
C_BILL_AMT6,0.0044
C_PAY_AMT1,0.1018
C_PAY_AMT2,0.0735
C_PAY_AMT3,0.0869
C_PAY_AMT4,0.0639


3 bins


,iv
C_BILL_AMT1,0.0096
C_BILL_AMT2,0.0055
C_BILL_AMT3,0.0063
C_BILL_AMT4,0.0058
C_BILL_AMT5,0.0048
C_BILL_AMT6,0.0008
C_PAY_AMT1,0.1661
C_PAY_AMT2,0.1055
C_PAY_AMT3,0.1057
C_PAY_AMT4,0.0856


4 bins


...

,iv
C_BILL_AMT1,0.0150
C_BILL_AMT2,0.0168
C_BILL_AMT3,0.0139
C_BILL_AMT4,0.0151
C_BILL_AMT5,0.0174
C_BILL_AMT6,0.0138
C_PAY_AMT1,0.1686
C_PAY_AMT2,0.1160
C_PAY_AMT3,0.1240
C_PAY_AMT4,0.0930


8 bins


,iv
C_BILL_AMT1,0.0220
C_BILL_AMT2,0.0162
C_BILL_AMT3,0.0255
C_BILL_AMT4,0.0190
C_BILL_AMT5,0.0181
C_BILL_AMT6,0.0178
C_PAY_AMT1,0.1742
C_PAY_AMT2,0.1163
C_PAY_AMT3,0.1339
C_PAY_AMT4,0.0980


9 bins


,iv
C_BILL_AMT1,0.0156
C_BILL_AMT2,0.0171
C_BILL_AMT3,0.0144
C_BILL_AMT4,0.0180
C_BILL_AMT5,0.0173
C_BILL_AMT6,0.0152
C_PAY_AMT1,0.1765
C_PAY_AMT2,0.1209
C_PAY_AMT3,0.1407
C_PAY_AMT4,0.1135


In [13]:
dic_bins = {}
for caracteristica in ls_continuas:
    df[f"C_{caracteristica}"] = pd.qcut(df[caracteristica], q=6, duplicates="drop").cat.add_categories(["Missing"]).fillna("Missing")
    bins = pd.qcut(df[caracteristica], q=6, duplicates="drop", retbins=True)
    dic_bins[caracteristica] = bins[1]

In [14]:
df

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,C_BILL_AMT1,C_BILL_AMT2,C_BILL_AMT3,C_BILL_AMT4,C_BILL_AMT5,C_BILL_AMT6,C_PAY_AMT1,C_PAY_AMT2,C_PAY_AMT3,C_PAY_AMT4,C_PAY_AMT5,C_PAY_AMT6,C_LIMIT_BAL,C_AGE
0,200000.0000,2,3,1,41,-1,-1,-1,-1,-1,2,3000.0000,187.0000,1306.0000,3142.0000,3332.0000,1686.0000,187.0000,1306.0000,3332.0000,1686.0000,0.0000,1496.0000,0,"(1129.667, 8601.0]","(-67526.001, 825.0]","(632.0, 7118.0]","(504.0, 6415.0]","(389.0, 5104.667]","(176.0, 4000.0]","(-0.001, 1472.0]","(-0.001, 1323.0]","(3016.333, 6145.0]","(1500.0, 3000.0]","(-0.001, 672.667]","(600.0, 1500.0]","(140000.0, 200000.0]","(39.0, 45.0]"
1,50000.0000,1,2,1,57,0,0,0,0,0,0,46540.0000,48520.0000,47875.0000,48110.0000,18576.0000,15339.0000,2770.0000,2100.0000,1300.0000,700.0000,1000.0000,1000.0000,0,"(22582.0, 48123.667]","(47099.0, 93416.0]","(44889.0, 89168.0]","(37838.333, 82123.333]","(17966.0, 32707.333]","(4000.0, 16496.0]","(2146.0, 4000.0]","(2044.0, 3680.333]","(1000.0, 1800.0]","(669.667, 1500.0]","(672.667, 1500.0]","(600.0, 1500.0]","(9999.999, 50000.0]","(45.0, 79.0]"
2,100000.0000,2,2,1,41,-1,-1,-1,-1,-1,-1,1594.0000,4612.0000,2098.0000,3494.0000,2086.0000,2379.0000,4612.0000,2099.0000,3494.0000,2086.0000,2379.0000,3219.0000,0,"(1129.667, 8601.0]","(825.0, 7611.333]","(632.0, 7118.0]","(504.0, 6415.0]","(389.0, 5104.667]","(176.0, 4000.0]","(4000.0, 7160.333]","(2044.0, 3680.333]","(3016.333, 6145.0]","(1500.0, 3000.0]","(1500.0, 3000.0]","(3000.0, 5843.667]","(80000.0, 140000.0]","(39.0, 45.0]"
3,380000.0000,1,3,1,44,0,0,0,0,0,0,329877.0000,309781.0000,295550.0000,277650.0000,264217.0000,238368.0000,12055.0000,9837.0000,9154.0000,9042.0000,8515.0000,7745.0000,1,"(98066.0, 746814.0]","(93416.0, 605943.0]","(89168.0, 577015.0]","(82123.333, 565669.0]","(77486.667, 524315.0]","(75169.0, 496915.0]","(7160.333, 873552.0]","(7009.667, 1215471.0]","(6145.0, 889043.0]","(5560.333, 621000.0]","(5773.0, 326889.0]","(5843.667, 527143.0]","(300000.0, 800000.0]","(39.0, 45.0]"
4,30000.0000,2,2,1,59,0,0,0,0,0,0,27620.0000,27921.0000,28585.0000,29007.0000,29195.0000,21335.0000,1449.0000,1503.0000,1315.0000,893.0000,1000.0000,513.0000,0,"(22582.0, 48123.667]","(20830.0, 47099.0]","(19953.0, 44889.0]","(19079.0, 37838.333]","(17966.0, 32707.333]","(16496.0, 30497.333]","(-0.001, 1472.0]","(1323.0, 2044.0]","(1000.0, 1800.0]","(669.667, 1500.0]","(672.667, 1500.0]","(-0.001, 600.0]","(9999.999, 50000.0]","(45.0, 79.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5620,170000.0000,1,1,2,27,2,0,0,0,2,2,46899.0000,45809.0000,44572.0000,43211.0000,41394.0000,37958.0000,2732.0000,2578.0000,2300.0000,2000.0000,1200.0000,0.0000,0,"(22582.0, 48123.667]","(20830.0, 47099.0]","(19953.0, 44889.0]","(37838.333, 82123.333]","(32707.333, 77486.667]","(30497.333, 75169.0]","(2146.0, 4000.0]","(2044.0, 3680.333]","(1800.0, 3016.333]","(1500.0, 3000.0]","(672.667, 1500.0]","(-0.001, 600.0]","(140000.0, 200000.0]","(20.999, 27.0]"
5621,200000.0000,2,1,2,37,-1,-1,-1,-1,0,-1,11211.0000,12329.0000,11846.0000,25920.0000,8035.0000,8521.0000,12329.0000,11846.0000,25924.0000,161.0000,8521.0000,5538.0000,0,"(8601.0, 22582.0]","(7611.333, 20830.0]","(7118.0, 19953.0]","(19079.0, 37838.333]","(5104.667, 17966.0]","(4000.0, 16496.0]","(7160.333, 873552.0]","(7009.667, 1215471.0]","(6145.0, 889043.0]","(-0.001, 669.667]","(5773.0, 326889.0]","(3000.0, 5843.667]","(140000.0, 200000.0]","(34.0, 39.0]"
5622,50000.0000,1,1,2,34,0,0,0,0,0,0,48559.0000,47008.0000,15435.0000,10235.0000,9119.0000,8237.0000,2573.0000,2000.0000,500.0000,459.0000,500.0000,500.0000,0,"(48123.667, 98066.0]","(20830.0, 47099.0]","(7118.0, 19953.0]","(6415.0, 19079.0]","(5104.667, 17966.0]","(4000.0, 16496.0]","(2146.0, 4000.0]","(1323

### IV

In [15]:
ls_discretas

['SEX',
 'EDUCATION',
 'MARRIAGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6']

> Creamos nuevos grupos en base a los anteriores.

* __(-1)__: Cliente cumplido --no uso o pago antes de la fecha--

* __(0)__: Cliente al corriente --uso normal sin atraso--

* __(1)__: Atraso leve --1 a 2 meses de retraso--
 
* __(2)__: Atraso moderado --3 a 4 meses de retraso--

* __(3)__: Atraso severo --5 o más meses de retraso--


In [16]:
redundantes_e = [0,5,6]
df['EDUCATION'] = df['EDUCATION'].replace(redundantes_e, 4)

for i in [0,2,3,4,5,6]:
    df[f"PAY_{i}"] = df[f"PAY_{i}"].replace(-2, -1)
    df[f"PAY_{i}"] = df[f"PAY_{i}"].replace(2, 1)
    df[f"PAY_{i}"] = df[f"PAY_{i}"].replace([4, 3],2)
    df[f"PAY_{i}"] = df[f"PAY_{i}"].replace([5,6,7,8,9], 3)

In [17]:
df_iv = pd.DataFrame(columns=["iv"])
for caracteristica in ls_discretas_C + ls_discretas:
    df_iv.loc[caracteristica, "iv"] = IV(df = df, var = caracteristica, tgt = target)

/var/folders/g0/w597jcd51wb8qzz27zv2xlr80000gn/T/ipykernel_48296/3673014158.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])
/var/folders/g0/w597jcd51wb8qzz27zv2xlr80000gn/T/ipykernel_48296/3673014158.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])
/var/folders/g0/w597jcd51wb8qzz27zv2xlr80000gn/T/ipykernel_48296/3673014158.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior o

In [18]:
df_iv.sort_values(by="iv", ascending=False)

,iv
PAY_0,0.7630
PAY_2,0.5942
PAY_3,0.4629
PAY_4,0.3887
PAY_5,0.3527
PAY_6,0.3046
C_LIMIT_BAL,0.1745
C_PAY_AMT1,0.1692
C_PAY_AMT2,0.1084
C_PAY_AMT3,0.1081


In [19]:
ls_iv_mejores = df_iv[df_iv["iv"]>0.1].index.tolist()

In [20]:
ls_iv_mejores

['C_PAY_AMT1',
 'C_PAY_AMT2',
 'C_PAY_AMT3',
 'C_LIMIT_BAL',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6']

### WOE

In [21]:
dic_woe = {}
for var in ls_iv_mejores:
    df, dic = WOE(df = df, var = var, tgt = target)
    dic_woe[var] = dic

/var/folders/g0/w597jcd51wb8qzz27zv2xlr80000gn/T/ipykernel_48296/822786008.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])


default.payment.next.month      evento no_evento %evento  \
                                          count  sum                            
C_PAY_AMT1                                                                      
(-0.001, 1472.0]                           1876  592    592      1284  0.4691   
(1472.0, 2146.0]                            937  194    194       743  0.1537   
(2146.0, 4000.0]                            995  216    216       779  0.1712   
(4000.0, 7160.333]                          879  137    137       742  0.1086   
(7160.333, 873552.0]                        938  123    123       815  0.0975   
Missing                                       0    0      0         0  0.0000   

                     %no_evento     WOE  
                                         
C_PAY_AMT1                               
(-0.001, 1472.0]         0.2943 -0.4662  
(1472.0, 2146.0]         0.1703  0.1024  
(2146.0, 4000.0]         0.1785  0.0423  
(4000.0, 7160.333]       0.1701  0.4489  
(7160.333, 873552.0]     0.1868  0.6505  
Missing                  0.0000     NaN

/var/folders/g0/w597jcd51wb8qzz27zv2xlr80000gn/T/ipykernel_48296/822786008.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])


default.payment.next.month      evento no_evento  \
                                           count  sum                    
C_PAY_AMT2                                                               
(-0.001, 1323.0]                            1876  551    551      1325   
(1323.0, 2044.0]                             937  212    212       725   
(2044.0, 3680.333]                           937  205    205       732   
(3680.333, 7009.667]                         937  159    159       778   
(7009.667, 1215471.0]                        938  135    135       803   
Missing                                        0    0      0         0   

                      %evento %no_evento     WOE  
                                                  
C_PAY_AMT2                                        
(-0.001, 1323.0]       0.4366     0.3037 -0.3630  
(1323.0, 2044.0]       0.1680     0.1662 -0.0109  
(2044.0, 3680.333]     0.1624     0.1678  0.0323  
(3680.333, 7009.667]   0.1260     0.1783  0.3474  
(7009.667, 1215471.0]  0.1070     0.1840  0.5426  
Missing                0.0000     0.0000     NaN

/var/folders/g0/w597jcd51wb8qzz27zv2xlr80000gn/T/ipykernel_48296/822786008.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  aux = df[[var, tgt]].groupby(var).agg(["count", "sum"])


...

default.payment.next.month      evento no_evento %evento %no_evento  \
                           count  sum                                       
PAY_2                                                                       
-1                          1887  315    315      1572  0.2496     0.3603   
 0                          2913  467    467      2446  0.3700     0.5606   
 1                           723  417    417       306  0.3304     0.0701   
 2                            89   54     54        35  0.0428     0.0080   
 3                            13    9      9         4  0.0071     0.0009   

          WOE  
               
PAY_2          
-1     0.3671  
 0     0.4154  
 1    -1.5500  
 2    -1.6741  
 3    -2.0514

default.payment.next.month      evento no_evento %evento %no_evento  \
                           count  sum                                       
PAY_3                                                                       
-1                          1899  307    307      1592  0.2433     0.3649   
 0                          2921  518    518      2403  0.4105     0.5508   
 1                           728  395    395       333  0.3130     0.0763   
 2                            63   32     32        31  0.0254     0.0071   
 3                            14   10     10         4  0.0079     0.0009   

          WOE  
               
PAY_3          
-1     0.4054  
 0     0.2940  
 1    -1.4112  
 2    -1.2722  
 3    -2.1568

default.payment.next.month      evento no_evento %evento %no_evento  \
                           count  sum                                       
PAY_4                                                                       
-1                          1896  319    319      1577  0.2528     0.3614   
 0                          3079  583    583      2496  0.4620     0.5721   
 1                           580  312    312       268  0.2472     0.0614   
 2                            43   28     28        15  0.0222     0.0034   
 3                            27   20     20         7  0.0158     0.0016   

          WOE  
               
PAY_4          
-1     0.3576  
 0     0.2138  
 1    -1.3925  
 2    -1.8646  
 3    -2.2903

default.payment.next.month      evento no_evento %evento %no_evento  \
                           count  sum                                       
PAY_5                                                                       
-1                          1938  331    331      1607  0.2623     0.3683   
 0                          3135  617    617      2518  0.4889     0.5771   
 1                           488  270    270       218  0.2139     0.0500   
 2                            45   29     29        16  0.0230     0.0037   
 3                            19   15     15         4  0.0119     0.0009   

          WOE  
               
PAY_5          
-1     0.3395  
 0     0.1659  
 1    -1.4544  
 2    -1.8352  
 3    -2.5622

default.payment.next.month      evento no_evento %evento %no_evento  \
                           count  sum                                       
PAY_6                                                                       
-1                          2026  359    359      1667  0.2845     0.3821   
 0                          3037  603    603      2434  0.4778     0.5579   
 1                           495  252    252       243  0.1997     0.0557   
 2                            50   33     33        17  0.0261     0.0039   
 3                            17   15     15         2  0.0119     0.0005   

          WOE  
               
PAY_6          
-1     0.2950  
 0     0.1549  
 1    -1.2768  
 2    -1.9038  
 3    -3.2554

In [22]:
ls_woe = [x for x in df.columns if x.startswith("W_")]

In [23]:
ls_woe

['W_C_PAY_AMT1',
 'W_C_PAY_AMT2',
 'W_C_PAY_AMT3',
 'W_C_LIMIT_BAL',
 'W_PAY_0',
 'W_PAY_2',
 'W_PAY_3',
 'W_PAY_4',
 'W_PAY_5',
 'W_PAY_6']

### Train-test split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(df[ls_woe], df[target])

### Regresión Logistica

In [25]:
rl = LogisticRegression()

In [26]:
rl.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [27]:
rl_scores = cross_val_score(X=X_train, y=y_train, cv=4, n_jobs=-1, estimator=rl, scoring="roc_auc")

In [28]:
rl_scores.mean(), rl_scores.std()

(np.float64(0.7576555908057099), np.float64(0.013638980309725912))

In [29]:
rl.score(X_test, y_test)

0.7974413646055437

In [30]:
roc_auc_score(y_score=rl.predict(X_test), y_true=y_test)

0.6209232375459879

## Predicción

### Carga de datos

In [31]:
df_pred = pd.read_csv("Clasificacion/val_default.csv", sep="|")

In [32]:
df_pred

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,6987,100000.0000,2,2,1,36,0,0,0,0,0,0,97026.0000,96109.0000,97647.0000,91998.0000,69915.0000,72044.0000,5000.0000,5000.0000,3500.0000,3000.0000,3000.0000,5049.0000
1,8727,50000.0000,1,2,2,38,1,2,2,2,0,0,15116.0000,14876.0000,17325.0000,16754.0000,17175.0000,17588.0000,300.0000,3000.0000,0.0000,1000.0000,690.0000,650.0000
2,24669,110000.0000,1,1,2,30,1,2,-1,-1,-1,0,2932.0000,2373.0000,2373.0000,1475.0000,3102.0000,3151.0000,0.0000,2373.0000,1475.0000,4000.0000,3000.0000,2373.0000
3,17951,330000.0000,2,2,1,30,0,0,0,0,0,-1,155285.0000,156122.0000,11733.0000,9620.0000,0.0000,4520.0000,10000.0000,10000.0000,2000.0000,0.0000,4520.0000,2500.0000
4,22159,430000.0000,1,2,2,30,-1,-1,2,0,0,0,28966.0000,29957.0000,29199.0000,30267.0000,49268.0000,48241.0000,5011.0000,0.0000,2000.0000,20000.0000,15000.0000,5000.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870,24917,500000.0000,2,2,2,32,0,-1,-1,-1,0,0,13631.0000,3840.0000,7527.0000,77166.0000,74397.0000,77055.0000,3840.0000,7527.0000,77166.0000,3397.0000,4055.0000,2266.0000
1871,6267,50000.0000,1,2,2,22,0,0,0,0,-1,0,51159.0000,51699.0000,50584.0000,20480.0000,9413.0000,9712.0000,3000.0000,2650.0000,2100.0000,10000.0000,600.0000,500.0000
1872,21721,160000.0000,2,2,2,24,0,0,0,0,0,0,55559.0000,51972.0000,34983.0000,26514.0000,17602.0000,9308.0000,3010.0000,3000.0000,2000.0000,1000.0000,2000.0000,0.0000
1873,11194,210000.0000,2,1,1,43,1,-2,-2,-2,-2,-2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [33]:
ids = df_pred["CUSTOMER_ID"]

In [34]:
for caracteristica in ls_continuas:
    df_pred[f"C_{caracteristica}"] = pd.cut(df_pred[caracteristica], bins=dic_bins[caracteristica], include_lowest=True).cat.add_categories(["Missing"]).fillna("Missing")

> Cambio de agrupación

In [35]:
redundantes_e = [0,5,6]
df_pred['EDUCATION'] = df_pred['EDUCATION'].replace(redundantes_e, 4)

for i in [0,2,3,4,5,6]:
    df_pred[f"PAY_{i}"] = df_pred[f"PAY_{i}"].replace(-2, -1)
    df_pred[f"PAY_{i}"] = df_pred[f"PAY_{i}"].replace(2, 1)
    df_pred[f"PAY_{i}"] = df_pred[f"PAY_{i}"].replace([4, 3],2)
    df_pred[f"PAY_{i}"] = df_pred[f"PAY_{i}"].replace([5,6,7,8,9], 3)

In [36]:
for var in ls_iv_mejores:
    df_pred["W_" + var] = df_pred[var].map(dic_woe[var])

### Predicción

In [37]:
df_prediccion = df_pred[["W_" + var for var in ls_iv_mejores]]

In [38]:
rl = LogisticRegression()

In [39]:
rl.fit(df[ls_woe], df[target])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [40]:
pred = rl.predict(df_prediccion)

In [41]:
df_resultado = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred
})

In [42]:
df_resultado

,CARDHOLDER_ID,y_hat
0,6987,0
1,8727,1
2,24669,0
3,17951,0
4,22159,0
...,...,...
1870,24917,0
1871,6267,0
1872,21721,0
1873,11194,0


In [43]:
df_resultado.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_CreditScoring.csv", index=False)